# Solve 1BRC — download data, aggregate, print result

End-to-end version of the challenge in one notebook: generate the input file if it isn't already there, read it with an explicit schema, aggregate per station, and print the challenge-spec result.

A `SparkSession` is already created for you as `spark` (see `00_getting_started.ipynb`). Defaults to the full 1 billion rows / 1000 stations — lower `ROWS`/`STATIONS` in the next cell for a quick test run before committing to the full size (generation alone takes several minutes at 1B rows).

In [ ]:
import os
import sys
import time

DATA_PATH = os.path.expanduser("~/data/measurements.txt")
ROWS = 1_000_000_000
STATIONS = 1_000
SEED = 42

if not os.path.exists(DATA_PATH):
    # generator seeded onto the node at ~/1brc-data-gen (see ansible/roles/jupyter)
    sys.path.insert(0, os.path.expanduser("~/1brc-data-gen"))
    from generate_measurements import load_stations, generate, DEFAULT_STATIONS_FILE

    os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
    picked = load_stations(DEFAULT_STATIONS_FILE, STATIONS, SEED)
    start = time.time()
    generate(DATA_PATH, ROWS, picked, SEED)
    print(f"generated {ROWS:,} rows in {time.time() - start:.1f}s")
else:
    print(f"{DATA_PATH} already exists, skipping generation")

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

MEASUREMENT_SCHEMA = StructType([
    StructField("station", StringType(), False),
    StructField("temperature", DoubleType(), False),
])

df = spark.read.csv(DATA_PATH, sep=";", schema=MEASUREMENT_SCHEMA, header=False)
df.printSchema()

In [ ]:
from pyspark.sql import functions as F

agg = df.groupBy("station").agg(
    F.min("temperature").alias("min_temp"),
    F.avg("temperature").alias("mean_temp"),
    F.max("temperature").alias("max_temp"),
    F.count("*").alias("n"),
).orderBy("station")

agg.explain()  # look for HashAggregate (partial) before the shuffle Exchange

In [ ]:
start = time.time()
rows = agg.collect()  # small: one row per station, not per input row
elapsed = time.time() - start

# f"{x:.1f}" (round-half-to-even) rather than Spark's round() (HALF_UP),
# to match tools/verify_sample.py exactly.
result = "{" + ", ".join(
    f"{r.station}={r.min_temp:.1f}/{r.mean_temp:.1f}/{r.max_temp:.1f}" for r in rows
) + "}"
total_rows = sum(r.n for r in rows)

print(f"rows={total_rows:,}  stations={len(rows)}  elapsed={elapsed:.1f}s")
print(result)

with open(os.path.expanduser("~/results.txt"), "w") as f:
    f.write(result + "\n")